In [4]:
                              # Part 3

In [5]:
# Task 3.1

In [6]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv("appointments_clean.csv")

In [7]:
# Selecting features and target

X = df[
    ["age", "distance_km", "fee_inr", "lead_days",
     "prior_no_shows", "reminder_sent",
     "booking_channel", "visit_type"]
].copy()

X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

y = df["no_show"]

In [8]:
# Creating the train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [9]:
print("Number of features:", X.shape[1])

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])

print("No-shows in test set:", y_test.sum())

baseline_accuracy = (y_test == 0).mean()
print("Baseline accuracy:", baseline_accuracy)

Number of features: 11
Training set size: 792
Testing set size: 198
No-shows in test set: 49
Baseline accuracy: 0.7525252525252525


In [10]:
# Task 3.2

In [20]:
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [21]:
# Scaling the features

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [22]:
# Training the Logistic Regression model

model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

LogisticRegression(max_iter=1000)

In [23]:
# Making predictions

y_pred = model.predict(X_test_s)

y_prob = model.predict_proba(X_test_s)[:, 1]

In [24]:
# Calculating evaluation metrics

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nPrecision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.7878787878787878

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1 Score: 0.4473684210526316
ROC-AUC: 0.788933022873579


In [25]:
# Displaying the model coefficients

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print(coefficients)

                    Feature  Coefficient
0                       age    -0.253880
1               distance_km     0.253448
2                   fee_inr     0.452761
3                 lead_days     0.614471
4            prior_no_shows     0.649612
5             reminder_sent    -0.580669
6       booking_channel_App    -0.046804
7     booking_channel_Phone     0.215881
8   booking_channel_Walk-In    -0.186364
9   visit_type_Consultation     0.015658
10     visit_type_Follow-up    -0.015658


In [26]:
# Interpreting the reminder_sent coefficient

reminder_index = X.columns.get_loc("reminder_sent")

reminder_coef = model.coef_[0][reminder_index]
reminder_scale = scaler.scale_[reminder_index]

odds_per_standard_deviation = np.exp(reminder_coef)

odds_no_to_yes = np.exp(reminder_coef / reminder_scale)

print("Reminder coefficient:", reminder_coef)
print("Odds ratio per standard deviation:", odds_per_standard_deviation)
print("Odds ratio for No to Yes:", odds_no_to_yes)

Reminder coefficient: -0.5806686033659922
Odds ratio per standard deviation: 0.559524141750829
Odds ratio for No to Yes: 0.3093980903124433


In [27]:
# Interpretation

# The model shows that reminder_sent has a negative association with no-shows, while lead_days and prior_no_shows have positive associations. The odds ratio for changing the reminder status from No to Yes is approximately 0.31, indicating about 69% lower odds of a no-show, holding other features constant. However, this is an association in observational data and does not prove that sending reminders causes fewer no-shows.

In [28]:
# Task 3.3

In [29]:
from sklearn.tree import DecisionTreeClassifier

In [30]:
# Training a decision tree without a depth limit

dt_full = DecisionTreeClassifier(random_state=42)

dt_full.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [31]:
# Calculating training and testing accuracy

train_pred_full = dt_full.predict(X_train)
test_pred_full = dt_full.predict(X_test)

print("Full Decision Tree")
print("Training Accuracy:", accuracy_score(y_train, train_pred_full))
print("Testing Accuracy:", accuracy_score(y_test, test_pred_full))
print("Tree Depth:", dt_full.get_depth())

Full Decision Tree
Training Accuracy: 1.0
Testing Accuracy: 0.6717171717171717
Tree Depth: 15


In [33]:
# Training a decision tree with maximum depth of 3

dt_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)

dt_depth3.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=3, random_state=42)

In [34]:
# Calculate training and testing accuracy

train_pred_depth3 = dt_depth3.predict(X_train)
test_pred_depth3 = dt_depth3.predict(X_test)

print("Depth-3 Decision Tree")
print("Training Accuracy:", accuracy_score(y_train, train_pred_depth3))
print("Testing Accuracy:", accuracy_score(y_test, test_pred_depth3))
print("Tree Depth:", dt_depth3.get_depth())

Depth-3 Decision Tree
Training Accuracy: 0.821969696969697
Testing Accuracy: 0.7777777777777778
Tree Depth: 3


In [35]:
# Interpretation

# The full decision tree achieved very high training accuracy but much lower testing accuracy, indicating overfitting. The depth-3 tree performed more consistently on both datasets, suggesting that limiting the tree depth helped it generalize better to unseen data.

In [36]:
# Task 3.4

In [37]:
from sklearn.ensemble import RandomForestClassifier

In [38]:
# Training the Random Forest model

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [39]:
# Making predictions and evaluating the model


rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:, 1]

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("Random Forest Recall:", recall_score(y_test, rf_pred))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_prob))

Random Forest Accuracy: 0.7474747474747475
Random Forest Recall: 0.30612244897959184
Random Forest ROC-AUC: 0.7057252431173813


In [40]:
# Finding the top three important features

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.head(3))

       Feature  Importance
3    lead_days    0.219905
0          age    0.196788
1  distance_km    0.194739


In [41]:
# Creating the model comparison table

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, model.predict(X_test_s)),
        accuracy_score(y_test, dt_depth3.predict(X_test)),
        accuracy_score(y_test, rf.predict(X_test))
    ],
    "Recall": [
        recall_score(y_test, model.predict(X_test_s)),
        recall_score(y_test, dt_depth3.predict(X_test)),
        recall_score(y_test, rf.predict(X_test))
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, model.predict_proba(X_test_s)[:, 1]),
        roc_auc_score(y_test, dt_depth3.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
    ]
})

print(comparison.round(3))

                     Model  Accuracy  Recall  ROC-AUC
0      Logistic Regression     0.788   0.347    0.789
1  Decision Tree (Depth 3)     0.778   0.224    0.700
2            Random Forest     0.747   0.306    0.706


In [42]:
# Observation

# The Logistic Regression model achieved the highest ROC-AUC among the three models, indicating better overall ability to distinguish between attended appointments and no-shows. The Random Forest provided useful feature-importance insights, while the depth-3 Decision Tree offered a simpler model. However, the final model will be selected using cross-validation in Task 3.5.

In [43]:
# Task 3.5

In [44]:
from sklearn.model_selection import GridSearchCV

In [45]:
# Defining the parameter grid

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

In [46]:
# Creating and running GridSearchCV

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

rf_grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [3, 5, 8], 'n_estimators': [100, 200]},
             scoring='roc_auc')

In [47]:
print("Best Parameters:", rf_grid.best_params_)
print("Best Cross-Validation ROC-AUC:", rf_grid.best_score_)

Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best Cross-Validation ROC-AUC: 0.7673242251739405


In [48]:
# Choosing the final model using 5-fold cross-validation


from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Decision Tree with depth 3
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)

# Random Forest with 100 trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Calculating 5-fold cross-validation ROC-AUC
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

dt_cv_auc = cross_val_score(
    dt_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    rf_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv_auc = rf_grid.best_score_

# Displaying results
print("Logistic Regression:", lr_cv_auc)
print("Decision Tree:", dt_cv_auc)
print("Random Forest:", rf_cv_auc)
print("Tuned Random Forest:", tuned_rf_cv_auc)

Logistic Regression: 0.7737139899948249
Decision Tree: 0.6784494968661952
Random Forest: 0.7466024782933701
Tuned Random Forest: 0.7673242251739405


In [50]:
# Choosing the classification threshold

from sklearn.model_selection import cross_val_predict
# from sklearn.metrics import precision_score, recall_score, f1_score


# Generating out-of-fold probabilities using Logistic Regression
oof_probs = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Comparing thresholds
for threshold in [0.50, 0.35]:

    oof_flags = (oof_probs >= threshold).astype(int)

    precision = precision_score(y_train, oof_flags)
    recall = recall_score(y_train, oof_flags)
    f1 = f1_score(y_train, oof_flags)
    flagged = oof_flags.sum()

    print(f"\nThreshold: {threshold}")
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("Number of appointments flagged:", flagged)


Threshold: 0.5
Precision: 0.5753424657534246
Recall: 0.2485207100591716
F1 Score: 0.34710743801652894
Number of appointments flagged: 73

Threshold: 0.35
Precision: 0.512987012987013
Recall: 0.46745562130177515
F1 Score: 0.4891640866873065
Number of appointments flagged: 154


In [51]:
# Evaluating the final model on the test set

from sklearn.metrics import precision_score, recall_score, roc_auc_score

# Fit the final model on the training data
lr_pipe.fit(X_train, y_train)

# Predict probabilities for the test set
test_probs = lr_pipe.predict_proba(X_test)[:, 1]

# Apply the selected threshold
chosen_threshold = 0.35
test_flags = (test_probs >= chosen_threshold).astype(int)

# Evaluate the model
test_precision = precision_score(y_test, test_flags)
test_recall = recall_score(y_test, test_flags)
test_roc_auc = roc_auc_score(y_test, test_probs)

print("Final Model: Logistic Regression")
print("Selected Threshold:", chosen_threshold)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Number of Appointments Flagged:", test_flags.sum())
print("Test ROC-AUC:", test_roc_auc)

Final Model: Logistic Regression
Selected Threshold: 0.35
Test Precision: 0.5714285714285714
Test Recall: 0.5714285714285714
Number of Appointments Flagged: 49
Test ROC-AUC: 0.788933022873579


In [52]:
# Creating the final test risk scores DataFrame
test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_flags
})

# Saving as CSV
test_risk_scores.to_csv("test_risk_scores.csv", index=False)

# Verifying the saved output
print("CSV saved successfully!")
print("Shape:", test_risk_scores.shape)
print("\nColumns:", test_risk_scores.columns.tolist())
print("\nFirst 5 rows:")
print(test_risk_scores.head())

CSV saved successfully!
Shape: (198, 4)

Columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']

First 5 rows:
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0


In [ ]:
# Explanation

# The Logistic Regression model was selected because it achieved the highest 5-fold cross-validation ROC-AUC of 0.7737. A threshold of 0.35 was chosen because it provided a better F1 score (0.4892) and higher recall than the 0.50 threshold. On the test set, the model achieved a precision of 0.5714, recall of 0.5714 and ROC-AUC of 0.7889, flagging 49 appointments as high risk. The risk scores were saved in `test_risk_scores.csv`. In practice, the clinic should choose its threshold based on the number of follow-up calls the front desk can make in an evening compared with the cost of leaving appointment slots empty.